# Word2Vec, FastText and Doc2Vec

This notebook implements and evaluates three embedding-based text representation techniques for sentiment classification:

1. Word2Vec
2. FastText
3. Doc2Vec

The same fixed dataset split created in `01_eda_and_data_split.ipynb` is used:

- **72% Train** → fit embedding models and classifiers
- **8% Validation** → select hyperparameters/configurations
- **20% Test** → final evaluation only

The training and validation sets are not combined after model selection. This keeps the experimental procedure consistent with the BERT experiment in this project.

In [35]:
import numpy as np
import pandas as pd

from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

### Load train, validation and test datasets

In [3]:
train_df = pd.read_csv("D:/NLP-Projects/sentiment-analysis-imdb/data/processed/train.csv")

validation_df = pd.read_csv("D:/NLP-Projects/sentiment-analysis-imdb/data/processed/validation.csv")

test_df = pd.read_csv("D:/NLP-Projects/sentiment-analysis-imdb/data/processed/test.csv")

In [4]:
print("Train shape:      ", train_df.shape)
print("Validation shape: ", validation_df.shape)
print("Test shape:       ", test_df.shape)

Train shape:       (35698, 7)
Validation shape:  (3967, 7)
Test shape:        (9917, 7)


### Select embedding text and labels

In [5]:
X_train_embed = train_df["embedding_review"]
X_val_embed = validation_df["embedding_review"]
X_test_embed = test_df["embedding_review"]

y_train = train_df["sentiment"]
y_val = validation_df["sentiment"]
y_test = test_df["sentiment"]

### Convert text into tokens

In [6]:
X_train_tokens = X_train_embed.str.split().tolist()
X_val_tokens = X_val_embed.str.split().tolist()
X_test_tokens = X_test_embed.str.split().tolist()

In [12]:
def evaluate_model(y_test,y_pred, case):
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"{case} Accuracy : {accuracy:.4f}")
    print(f"{case} Precision: {precision:.4f}")
    print(f"{case} Recall   : {recall:.4f}")
    print(f"{case} F1 Score : {f1:.4f}")

## 1. Word2Vec

Word2Vec learns dense vector representations for individual words.

The basic idea is that words appearing in similar contexts tend to receive similar vector representations.

Word2Vec has two main training architectures:

- **CBOW (Continuous Bag of Words):** predicts a target word from surrounding context words.
- **Skip-gram:** predicts surrounding words from a target word.

Word2Vec produces a vector for each word, not directly for an entire document.

For sentiment classification, we therefore need to convert the word vectors into a fixed-size document vector.

A common approach is to calculate the mean of the word vectors appearing in each document.

In [7]:
# Document vector function
def document_vector(model, tokens):
    vectors = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]

    if not vectors:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [8]:
w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=20,
    sg=1,
    seed=42
)

In [9]:
X_train_w2v = np.array([
    document_vector(w2v_model, tokens)
    for tokens in X_train_tokens
])

In [10]:
X_val_w2v = np.array([
    document_vector(w2v_model, tokens)
    for tokens in X_val_tokens
])

In [11]:
w2v_classifier = LogisticRegression(
    max_iter=1000,
    random_state=42
)

w2v_classifier.fit(
    X_train_w2v,
    y_train
)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [14]:
y_val_w2v_pred = w2v_classifier.predict(X_val_w2v)

In [16]:
evaluate_model(y_val, y_val_w2v_pred, "validation")

validation Accuracy : 0.8666
validation Precision: 0.8637
validation Recall   : 0.8719
validation F1 Score : 0.8678


### Word2Vec - Hyperparameter tuning

In [62]:
w2v_configs = {
    "skipgram_100": {
        "vector_size": 100,
        "window": 5,
        "min_count": 2,
        "epochs": 20,
        "sg": 1
    },
    
    "skipgram_200": {
        "vector_size": 200,
        "window": 5,
        "min_count": 2,
        "epochs": 20,
        "sg": 1
    },
    
    "cbow_100": {
        "vector_size": 100,
        "window": 5,
        "min_count": 2,
        "epochs": 20,
        "sg": 0
    }
}

In [63]:
w2v_results = []

for name, params in tqdm(
    w2v_configs.items(),
    desc="Training Word2Vec configurations"
):

    print(f"\nCurrently training: {name}")

    progress = TrainingProgress()
    
    w2v_model = Word2Vec(
        sentences=X_train_tokens,
        vector_size=params["vector_size"],
        window=params["window"],
        min_count=params["min_count"],
        workers=4,
        epochs=params["epochs"],
        sg=params["sg"],
        seed=42
    )

    X_train_w2v = np.array([
        document_vector(w2v_model, tokens)
        for tokens in X_train_tokens
    ])

    X_val_w2v = np.array([
        document_vector(w2v_model, tokens)
        for tokens in X_val_tokens
    ])

    classifier = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    classifier.fit(X_train_w2v, y_train)

    y_val_pred = classifier.predict(X_val_w2v)

    w2v_results.append({
        "configuration": name,
        "accuracy": accuracy_score(y_val, y_val_pred),
        "precision": precision_score(y_val, y_val_pred),
        "recall": recall_score(y_val, y_val_pred),
        "f1": f1_score(y_val, y_val_pred),
        "vector_size": params["vector_size"],
        "window": params["window"],
        "min_count": params["min_count"],
        "epochs": params["epochs"],
        "sg": params["sg"],
    })

Training Word2Vec configurations:   0%|                                                          | 0/3 [00:00<?, ?it/s]


Currently training: skipgram_100


Training Word2Vec configurations:  33%|████████████████▎                                | 1/3 [03:47<07:35, 227.58s/it]


Currently training: skipgram_200


Training Word2Vec configurations:  67%|████████████████████████████████▋                | 2/3 [09:06<04:41, 281.36s/it]


Currently training: cbow_100


Training Word2Vec configurations: 100%|█████████████████████████████████████████████████| 3/3 [10:21<00:00, 207.22s/it]


In [92]:
pd.DataFrame(w2v_results).sort_values("f1", ascending = False)

,configuration,accuracy,precision,recall,f1,vector_size,window,min_count,epochs,sg,representation
1,skipgram_200,0.873708,0.866995,0.883978,0.875404,200,5,2,20,1,Word2Vec
0,skipgram_100,0.866650,0.862959,0.872928,0.867915,100,5,2,20,1,Word2Vec
2,cbow_100,0.852533,0.846305,0.862883,0.854514,100,5,2,20,0,Word2Vec


### Word2Vec Observations

- **Skip-gram performed better than CBOW** on the validation set for the configurations tested.
- Increasing the vector size from **100 to 200 dimensions** improved the validation performance for Skip-gram.
- The best Word2Vec configuration was **Skip-gram with 200 dimensions**, achieving a validation F1 score of approximately **0.878**.
- Since document vectors were created by averaging the word vectors, this representation does not explicitly preserve the complete word order of the review.

## 2. FastText

FastText is an extension of the word-embedding idea behind Word2Vec.

Instead of representing a word only as a single learned word vector, FastText also uses **character n-grams (subwords)**.

For example, a word such as:

    "playing"

can be represented using information from smaller character sequences.

This allows FastText to capture some information about:

- word morphology
- rare words
- unseen or unusual words

Like Word2Vec, FastText produces word vectors rather than complete document vectors.

We therefore create a document representation by aggregating the vectors of the words in each review.

In [18]:
def fasttext_document_vector(model, tokens):
    vectors = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]

    if not vectors:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [19]:
fasttext_model = FastText(
    sentences=X_train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=20,
    sg=1,
    seed=42
)

In [20]:
X_train_fasttext = np.array([
    fasttext_document_vector(fasttext_model, tokens)
    for tokens in X_train_tokens
])

X_val_fasttext = np.array([
    fasttext_document_vector(fasttext_model, tokens)
    for tokens in X_val_tokens
])

In [21]:
fasttext_classifier = LogisticRegression(
    max_iter=1000,
    random_state=42
)

fasttext_classifier.fit(
    X_train_fasttext,
    y_train
)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [23]:
y_val_fasttext_pred = fasttext_classifier.predict(
    X_val_fasttext
)

In [41]:
evaluate_model(y_val, y_val_fasttext_pred, "validation")

validation Accuracy : 0.8654
validation Precision: 0.8633
validation Recall   : 0.8694
validation F1 Score : 0.8664


### Fasttext - Hyperparameter tuning

In [42]:
fasttext_configs = {
    "skipgram_100": {
        "vector_size": 100,
        "window": 5,
        "min_count": 2,
        "epochs": 20,
        "sg": 1,
        "min_n": 3,
        "max_n": 6
    },

    "skipgram_200": {
        "vector_size": 200,
        "window": 5,
        "min_count": 2,
        "epochs": 20,
        "sg": 1,
        "min_n": 3,
        "max_n": 6
    },

    "cbow_100": {
        "vector_size": 100,
        "window": 5,
        "min_count": 2,
        "epochs": 20,
        "sg": 0,
        "min_n": 3,
        "max_n": 6
    },

    "skipgram_100_wide_window": {
        "vector_size": 100,
        "window": 10,
        "min_count": 2,
        "epochs": 20,
        "sg": 1,
        "min_n": 3,
        "max_n": 6
    }
}

In [45]:
from tqdm import tqdm

fasttext_results = []

for name, params in tqdm(
    fasttext_configs.items(),
    desc="FastText experiments"
):

    print(f"\nCurrently training: {name}")

    # Train FastText only on training data
    fasttext_model = FastText(
        sentences=X_train_tokens,
        vector_size=params["vector_size"],
        window=params["window"],
        min_count=params["min_count"],
        epochs=params["epochs"],
        sg=params["sg"],
        min_n=params["min_n"],
        max_n=params["max_n"],
        workers=4,
        seed=42
    )

    # Create document vectors
    X_train_fasttext = np.array([
        document_vector(fasttext_model, tokens)
        for tokens in X_train_tokens
    ])

    X_val_fasttext = np.array([
        document_vector(fasttext_model, tokens)
        for tokens in X_val_tokens
    ])

    # Train classifier
    classifier = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    classifier.fit(
        X_train_fasttext,
        y_train
    )

    # Validation prediction
    y_val_pred = classifier.predict(
        X_val_fasttext
    )

    # Store results
    fasttext_results.append({
        "configuration": name,
        "vector_size": params["vector_size"],
        "window": params["window"],
        "min_count": params["min_count"],
        "epochs": params["epochs"],
        "sg": params["sg"],
        "min_n": params["min_n"],
        "max_n": params["max_n"],
        "accuracy": accuracy_score(y_val, y_val_pred),
        "precision": precision_score(y_val, y_val_pred),
        "recall": recall_score(y_val, y_val_pred),
        "f1": f1_score(y_val, y_val_pred)
    })

FastText experiments:   0%|                                                                      | 0/4 [00:00<?, ?it/s]


Currently training: skipgram_100


FastText experiments:  25%|███████████████▎                                             | 1/4 [08:05<24:15, 485.22s/it]


Currently training: skipgram_200


FastText experiments:  50%|██████████████████████████████▌                              | 2/4 [19:26<20:00, 600.29s/it]


Currently training: cbow_100


FastText experiments:  75%|█████████████████████████████████████████████▊               | 3/4 [24:14<07:38, 458.00s/it]


Currently training: skipgram_100_wide_window


FastText experiments: 100%|█████████████████████████████████████████████████████████████| 4/4 [36:04<00:00, 541.06s/it]


In [93]:
pd.DataFrame(fasttext_results).sort_values("f1", ascending = False)

,configuration,vector_size,window,min_count,epochs,sg,min_n,max_n,accuracy,precision,recall,f1,representation
1,skipgram_200,200,5,2,20,1,3,6,0.872196,0.869522,0.876946,0.873218,Fasttext
3,skipgram_100_wide_window,100,10,2,20,1,3,6,0.871944,0.868722,0.877449,0.873063,Fasttext
0,skipgram_100,100,5,2,20,1,3,6,0.867658,0.865040,0.872426,0.868717,Fasttext
2,cbow_100,100,5,2,20,0,3,6,0.839425,0.831863,0.852336,0.841975,Fasttext


## FastText Observations

- **Skip-gram outperformed CBOW** for the FastText configurations tested.
- Increasing the vector size from **100 to 200 dimensions** improved validation performance for Skip-gram.
- Increasing the context window from **5 to 10** produced almost no improvement for the 100-dimensional Skip-gram configuration.
- The best FastText configuration was **Skip-gram with 200 dimensions, window=5, and character n-grams from 3 to 6**, with a validation F1 score of approximately **0.873**.


## 3. Doc2Vec

Doc2Vec extends the Word2Vec idea to learn a vector representation for an entire document.

Unlike Word2Vec and FastText, which learn word vectors and require us to aggregate those vectors into a document representation, Doc2Vec directly learns document vectors during training.

Doc2Vec has two main architectures:

- **PV-DM (Distributed Memory)**
- **PV-DBOW (Distributed Bag of Words)**

In Gensim:

    dm=1 → PV-DM
    dm=0 → PV-DBOW

Each training document receives a unique tag, which is associated with its learned document vector.

In [25]:
train_documents = [
    TaggedDocument(
        words=tokens,
        tags=[i]
    )
    for i, tokens in enumerate(X_train_tokens)
]

In [26]:
doc2vec_model = Doc2Vec(
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=20,
    dm=0,
    seed=42
)

In [27]:
doc2vec_model.build_vocab(train_documents)

doc2vec_model.train(
    train_documents,
    total_examples=doc2vec_model.corpus_count,
    epochs=doc2vec_model.epochs
)

In [28]:
X_train_d2v = np.array([
    doc2vec_model.dv[i]
    for i in range(len(train_documents))
])

In [29]:
X_val_d2v = np.array([
    doc2vec_model.infer_vector(tokens)
    for tokens in X_val_tokens
])

In [30]:
d2v_classifier = LogisticRegression(
    max_iter=1000,
    random_state=42
)

d2v_classifier.fit(
    X_train_d2v,
    y_train
)

y_val_d2v_pred = d2v_classifier.predict(
    X_val_d2v
)

In [32]:
evaluate_model(y_val, y_val_d2v_pred, "validation")

validation Accuracy : 0.8843
validation Precision: 0.8892
validation Recall   : 0.8790
validation F1 Score : 0.8841


### Doc2Vec - Hyperparameter tuning

In [46]:
doc2vec_configs = {
    "pv_dbow_100": {
        "vector_size": 100,
        "window": 5,
        "min_count": 2,
        "epochs": 20,
        "dm": 0
    },

    "pv_dbow_200": {
        "vector_size": 200,
        "window": 5,
        "min_count": 2,
        "epochs": 20,
        "dm": 0
    },

    "pv_dm_100": {
        "vector_size": 100,
        "window": 5,
        "min_count": 2,
        "epochs": 20,
        "dm": 1
    },

    "pv_dm_100_wide_window": {
        "vector_size": 100,
        "window": 10,
        "min_count": 2,
        "epochs": 20,
        "dm": 1
    }
}

In [47]:
doc2vec_results = []

for name, params in tqdm(
    doc2vec_configs.items(),
    desc="Doc2Vec experiments"
):

    print(f"\nCurrently training: {name}")

    # Train Doc2Vec only on training documents
    doc2vec_model = Doc2Vec(
        vector_size=params["vector_size"],
        window=params["window"],
        min_count=params["min_count"],
        epochs=params["epochs"],
        dm=params["dm"],
        workers=4,
        seed=42
    )

    doc2vec_model.build_vocab(train_documents)

    doc2vec_model.train(
        train_documents,
        total_examples=doc2vec_model.corpus_count,
        epochs=doc2vec_model.epochs
    )

    # Training document vectors
    X_train_d2v = np.array([
        doc2vec_model.dv[i]
        for i in range(len(train_documents))
    ])

    # Validation document vectors
    # Validation documents were not used to train Doc2Vec,
    # so their vectors must be inferred.
    X_val_d2v = np.array([
        doc2vec_model.infer_vector(tokens)
        for tokens in X_val_tokens
    ])

    # Train classifier
    classifier = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    classifier.fit(
        X_train_d2v,
        y_train
    )

    # Validation prediction
    y_val_pred = classifier.predict(
        X_val_d2v
    )

    # Store results
    doc2vec_results.append({
        "configuration": name,
        "vector_size": params["vector_size"],
        "window": params["window"],
        "min_count": params["min_count"],
        "epochs": params["epochs"],
        "dm": params["dm"],
        "accuracy": accuracy_score(y_val, y_val_pred),
        "precision": precision_score(y_val, y_val_pred),
        "recall": recall_score(y_val, y_val_pred),
        "f1": f1_score(y_val, y_val_pred)
    })

Doc2Vec experiments:   0%|                                                                       | 0/4 [00:00<?, ?it/s]


Currently training: pv_dbow_100


Doc2Vec experiments:  25%|███████████████▊                                               | 1/4 [01:10<03:32, 70.86s/it]


Currently training: pv_dbow_200


Doc2Vec experiments:  50%|███████████████████████████████▌                               | 2/4 [02:34<02:36, 78.27s/it]


Currently training: pv_dm_100


Doc2Vec experiments:  75%|███████████████████████████████████████████████▎               | 3/4 [04:24<01:32, 92.91s/it]


Currently training: pv_dm_100_wide_window


Doc2Vec experiments: 100%|███████████████████████████████████████████████████████████████| 4/4 [06:19<00:00, 94.76s/it]


### Doc2Vec Observations

- **PV-DBOW substantially outperformed PV-DM** for the configurations tested.
- The best configuration was **PV-DBOW with 100 dimensions, window=5, min_count=2, and 20 epochs**, achieving a validation F1 score of approximately **0.886**.
- Increasing the PV-DBOW vector size from **100 to 200 dimensions** slightly reduced validation performance.
- Increasing the PV-DM window from **5 to 10** produced only a very small improvement, but PV-DM remained substantially below PV-DBOW.
- Among the three embedding techniques tested, **Doc2Vec achieved the best validation performance**.

### Combining the validation results

In [65]:
for result in w2v_results:
    result["representation"] = "Word2Vec"

for result in fasttext_results:
    result["representation"] = "Fasttext"

for result in doc2vec_results:
    result["representation"] = "Doc2Vec"

In [66]:
all_results = w2v_results + fasttext_results + doc2vec_results

In [67]:
embedding_results = pd.DataFrame(all_results)

In [68]:
embedding_results = embedding_results.sort_values(
    "f1",
    ascending=False
).reset_index(drop=True)

embedding_results

,configuration,accuracy,precision,recall,f1,vector_size,window,min_count,epochs,sg,representation,min_n,max_n,dm
0,pv_dbow_100,0.885556,0.889114,0.881969,0.885527,100,5,2,20,NaN,Doc2Vec,NaN,NaN,0.0
1,pv_dbow_200,0.883287,0.881618,0.886489,0.884047,200,5,2,20,NaN,Doc2Vec,NaN,NaN,0.0
2,skipgram_200,0.873708,0.866995,0.883978,0.875404,200,5,2,20,1.0,Word2Vec,NaN,NaN,NaN
3,skipgram_200,0.872196,0.869522,0.876946,0.873218,200,5,2,20,1.0,Fasttext,3.0,6.0,NaN
4,skipgram_100_wide_window,0.871944,0.868722,0.877449,0.873063,100,10,2,20,1.0,Fasttext,3.0,6.0,NaN
5,skipgram_100,0.867658,0.865040,0.872426,0.868717,100,5,2,20,1.0,Fasttext,3.0,6.0,NaN
6,skipgram_100,0.866650,0.862959,0.872928,0.867915,100,5,2,20,1.0,Word2Vec,NaN,NaN,NaN
7,cbow_100,0.852533,0.846305,0.862883,0.854514,100,5,2,20,0.0,Word2Vec,NaN,NaN,NaN
8,cbow_100,0.839425,0.831863,0.852336,0.841975,100,5,2,20,0.0,Fasttext,3.0,6.0,NaN
9,pv_dm_100_wide_window,0.838165,0.830152,0.851833,0.840853,100,10,2,20,NaN,Doc2Vec,NaN,NaN,1.0


### Embedding Representation Comparison

Based on the validation experiments:

- **Doc2Vec** achieved the highest validation F1 score among the three embedding approaches.
- **Word2Vec** was the second-best representation.
- **FastText** produced the lowest validation F1 score among the three.
- Doc2Vec was therefore selected as the strongest embedding approach based on the validation results.
- The best configuration from each representation was carried forward for final evaluation on the test set.

### Get the Best configurations from validation results

In [72]:
w2v_results_df = pd.DataFrame(w2v_results)
fasttext_results_df = pd.DataFrame(fasttext_results)
doc2vec_results_df = pd.DataFrame(doc2vec_results)

In [73]:
best_w2v = w2v_results_df.loc[w2v_results_df["f1"].idxmax()]
best_fasttext = fasttext_results_df.loc[fasttext_results_df["f1"].idxmax()]
best_doc2vec = doc2vec_results_df.loc[doc2vec_results_df["f1"].idxmax()]

### Word2Vec - final test evaluation

In [76]:
best_w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=int(best_w2v["vector_size"]),
    window=int(best_w2v["window"]),
    min_count=int(best_w2v["min_count"]),
    workers=4,
    epochs=int(best_w2v["epochs"]),
    sg=int(best_w2v["sg"]),
    seed=42
)

X_train_w2v = np.array([
    document_vector(best_w2v_model, tokens)
    for tokens in X_train_tokens
])

X_test_w2v = np.array([
    document_vector(best_w2v_model, tokens)
    for tokens in X_test_tokens
])

w2v_classifier = LogisticRegression(
    max_iter=1000,
    random_state=42
)
w2v_classifier.fit(X_train_w2v, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [77]:

y_test_w2v_pred = w2v_classifier.predict(X_test_w2v)

In [78]:
evaluate_model(y_test, y_test_w2v_pred, "test")

test Accuracy : 0.8819
test Precision: 0.8791
test Recall   : 0.8867
test F1 Score : 0.8829


### Fasttext - final test evaluation

In [79]:
best_fasttext_model = FastText(
    sentences=X_train_tokens,
    vector_size=int(best_fasttext["vector_size"]),
    window=int(best_fasttext["window"]),
    min_count=int(best_fasttext["min_count"]),
    epochs=int(best_fasttext["epochs"]),
    sg=int(best_fasttext["sg"]),
    min_n=int(best_fasttext["min_n"]),
    max_n=int(best_fasttext["max_n"]),
    workers=4,
    seed=42
)

X_train_fasttext = np.array([
    document_vector(best_fasttext_model, tokens)
    for tokens in X_train_tokens
])

X_test_fasttext = np.array([
    document_vector(best_fasttext_model, tokens)
    for tokens in X_test_tokens
])

fasttext_classifier = LogisticRegression(
    max_iter=1000,
    random_state=42
)

fasttext_classifier.fit(
    X_train_fasttext,
    y_train
)

Exception ignored in: <function tqdm.__del__ at 0x00000218CC962980>
Traceback (most recent call last):
  File "C:\Users\its0r\AppData\Roaming\Python\Python313\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "C:\Users\its0r\AppData\Roaming\Python\Python313\site-packages\tqdm\notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [80]:
y_test_fasttext_pred = fasttext_classifier.predict(
    X_test_fasttext
)

In [87]:
evaluate_model(y_test, y_test_fasttext_pred, "test")

test Accuracy : 0.8819
test Precision: 0.8797
test Recall   : 0.8859
test F1 Score : 0.8828


In [81]:
train_documents = [
    TaggedDocument(
        words=tokens,
        tags=[i]
    )
    for i, tokens in enumerate(X_train_tokens)
]

### Doc2Vec - final test evaluation

In [82]:
best_doc2vec_model = Doc2Vec(
    vector_size=int(best_doc2vec["vector_size"]),
    window=int(best_doc2vec["window"]),
    min_count=int(best_doc2vec["min_count"]),
    epochs=int(best_doc2vec["epochs"]),
    dm=int(best_doc2vec["dm"]),
    workers=4,
    seed=42
)

best_doc2vec_model.build_vocab(train_documents)

best_doc2vec_model.train(
    train_documents,
    total_examples=best_doc2vec_model.corpus_count,
    epochs=best_doc2vec_model.epochs
)

In [83]:
X_train_d2v = np.array([
    best_doc2vec_model.dv[i]
    for i in range(len(train_documents))
])

In [84]:
X_test_d2v = np.array([
    best_doc2vec_model.infer_vector(tokens)
    for tokens in X_test_tokens
])

In [85]:
d2v_classifier = LogisticRegression(
    max_iter=1000,
    random_state=42
)

d2v_classifier.fit(
    X_train_d2v,
    y_train
)

y_test_d2v_pred = d2v_classifier.predict(
    X_test_d2v
)

In [88]:
evaluate_model(y_test, y_test_d2v_pred, "test")

test Accuracy : 0.8933
test Precision: 0.8982
test Recall   : 0.8881
test F1 Score : 0.8931


### Create final test results dataframe

In [86]:
embedding_test_results = pd.DataFrame({
    "representation": [
        "Word2Vec",
        "FastText",
        "Doc2Vec"
    ],

    "configuration": [
        best_w2v["configuration"],
        best_fasttext["configuration"],
        best_doc2vec["configuration"]
    ],

    "classifier": [
        "LogisticRegression",
        "LogisticRegression",
        "LogisticRegression"
    ],

    "accuracy": [
        accuracy_score(y_test, y_test_w2v_pred),
        accuracy_score(y_test, y_test_fasttext_pred),
        accuracy_score(y_test, y_test_d2v_pred)
    ],

    "precision": [
        precision_score(y_test, y_test_w2v_pred),
        precision_score(y_test, y_test_fasttext_pred),
        precision_score(y_test, y_test_d2v_pred)
    ],

    "recall": [
        recall_score(y_test, y_test_w2v_pred),
        recall_score(y_test, y_test_fasttext_pred),
        recall_score(y_test, y_test_d2v_pred)
    ],

    "f1": [
        f1_score(y_test, y_test_w2v_pred),
        f1_score(y_test, y_test_fasttext_pred),
        f1_score(y_test, y_test_d2v_pred)
    ]
})

embedding_test_results.sort_values(
    "f1",
    ascending=False
).reset_index(drop=True)

,representation,configuration,classifier,accuracy,precision,recall,f1
0,Doc2Vec,pv_dbow_100,LogisticRegression,0.893315,0.898191,0.888085,0.893110
1,Word2Vec,skipgram_200,LogisticRegression,0.881920,0.879084,0.886679,0.882865
2,FastText,skipgram_200,LogisticRegression,0.881920,0.879689,0.885875,0.882771


### Final Test Observations

- **Doc2Vec PV-DBOW (100 dimensions)** achieved the best test performance among the embedding-based approaches, with an F1 score of approximately **0.893** and accuracy of approximately **0.893**.
- **Word2Vec Skip-gram (200 dimensions)** achieved an F1 score of approximately **0.883**.
- **FastText Skip-gram (200 dimensions)** achieved an F1 score of approximately **0.883**, making Word2Vec and FastText effectively tied in this experiment.
- The ranking observed on the validation set was preserved on the test set: **Doc2Vec > Word2Vec > FastText**.
- The test set was used only for final evaluation after the configurations had been selected using the validation set.

### Saving the results

In [89]:
embedding_test_results.to_csv(
    "D:/NLP-Projects/sentiment-analysis-imdb/results/word_embeddings_results.csv",
    index=False
)

# Conclusion

Three embedding-based text representations were evaluated for sentiment classification using Logistic Regression:

| Representation | Best configuration | Test F1 |
|---|---|---:|
| **Doc2Vec** | PV-DBOW, 100 dimensions | **0.8931** |
| Word2Vec | Skip-gram, 200 dimensions | 0.8829 |
| FastText | Skip-gram, 200 dimensions | 0.8828 |

**Doc2Vec PV-DBOW performed best among the embedding-based representations tested.**

The model configurations were selected using the fixed validation set, while the test set was reserved for final evaluation. The training and validation sets were not combined before the final test evaluation, maintaining consistency with the BERT experiment.